# ML4SCI EXXA — ALMA Protoplanetary Disk Autoencoder
**GSoC 2026 Test Submission | Arya Wadhwa (@aryawadhwa)**

This notebook satisfies both the **General Test** (unsupervised clustering) and the **Image-Based Test** (autoencoder with accessible latent space) for the EXXA subprojects.

### Pipeline
```
FITS files (1250μm continuum, 600×600)
    → log-stretch + min-max normalisation
    → ConvAutoencoder (5-stage, latent_dim=256)
    → Training (MSE + MS-SSIM loss)
    → Latent extraction
    → UMAP / t-SNE dimensionality reduction
    → KMeans clustering
    → Cluster gallery + radial profiles
```

**Key metrics:** MSE between input/output, Multi-scale SSIM, cluster silhouette score.

## 1. Install Dependencies

In [ ]:
!pip install -q astropy pytorch-msssim umap-learn tqdm scikit-learn matplotlib scipy

In [ ]:
import torch
print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 2. Mount Google Drive & Point to Data

Upload the `continuum_data_subset` folder to Google Drive, then update `FITS_DIR` below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── UPDATE THIS PATH to where you uploaded the FITS folder on Drive ──
FITS_DIR = '/content/drive/MyDrive/continuum_data_subset'

import os
fits_files = [f for f in os.listdir(FITS_DIR) if f.endswith('.fits')]
print(f"Found {len(fits_files)} FITS files")
print("Sample:", fits_files[:5])

## 3. Dataset & Preprocessing

In [ ]:
import numpy as np
from pathlib import Path
from astropy.io import fits as astrofits
import torch
from torch.utils.data import Dataset, DataLoader, random_split

class ALMADataset(Dataset):
    """Loads ALMA continuum FITS cubes (1250 micron, 600x600).
    
    Each file is a 4-layer cube; only layer 0 (Stokes I) is used,
    as specified in the EXXA test instructions.
    """
    def __init__(self, fits_dir, preprocess='log', normalize=True):
        self.paths = sorted(Path(fits_dir).glob('*.fits'))
        self.preprocess = preprocess
        self.normalize = normalize
        print(f"Dataset: {len(self.paths)} files")

    def __len__(self): return len(self.paths)

    def _load(self, path):
        with astrofits.open(path, memmap=False) as hdul:
            data = hdul[0].data.squeeze().astype(np.float32)
        # Use layer 0 as specified (Stokes I continuum)
        if data.ndim == 3: data = data[0]
        data = np.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0)
        data = np.clip(data, 0.0, None)
        # log-stretch (DeepFocus convention — compresses bright continuum,
        # brings up faint gap/ring substructure)
        if self.preprocess == 'log': data = np.log1p(data)
        if self.normalize:
            lo, hi = data.min(), data.max()
            data = (data - lo) / (hi - lo + 1e-8)
        return torch.tensor(data, dtype=torch.float32).unsqueeze(0)  # (1, H, W)

    def __getitem__(self, idx):
        return self._load(self.paths[idx]), str(self.paths[idx].name)

dataset = ALMADataset(FITS_DIR)
n_val = max(1, int(0.15 * len(dataset)))
n_train = len(dataset) - n_val
train_ds, val_ds = random_split(dataset, [n_train, n_val],
                                generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=2, pin_memory=True)
print(f"Train: {n_train} | Val: {n_val}")

## 4. Visualise Sample Images

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for i, ax in enumerate(axes.flat):
    img, name = dataset[i]
    # Parse planet count from filename (e.g. 'planet3_...' = 3 planets)
    n_planets = name.split('_')[0].replace('planet', '')
    ax.imshow(img[0].numpy(), cmap='inferno', origin='lower')
    ax.set_title(f'N={n_planets} planets', fontsize=8)
    ax.axis('off')
plt.suptitle('Sample ALMA 1250μm Continuum Images (log-stretch)', fontsize=13)
plt.tight_layout()
plt.savefig('sample_images.png', dpi=150)
plt.show()

## 5. Model: Convolutional Autoencoder

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

_BFLAT = 512 * 19 * 19  # 184,832

class EncoderBlock(nn.Sequential):
    def __init__(self, ic, oc):
        super().__init__(
            nn.Conv2d(ic, oc, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(oc),
            nn.LeakyReLU(0.2, inplace=True))

class DecoderBlock(nn.Sequential):
    def __init__(self, ic, oc, op=0):
        super().__init__(
            nn.ConvTranspose2d(ic, oc, 4, stride=2, padding=1, output_padding=op, bias=False),
            nn.BatchNorm2d(oc),
            nn.ReLU(inplace=True))

class ConvAutoencoder(nn.Module):
    """5-stage CAE for 1x600x600 ALMA images.
    
    Spatial flow:
        Encoder: 600→300→150→75→38→19 (stride-2 each step)
        Latent:  FC 184832→latent_dim with LayerNorm
        Decoder: FC latent_dim→184832 then 19→38→75→150→300→600
    """
    def __init__(self, latent_dim=256):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(
            EncoderBlock(1,   32),
            EncoderBlock(32,  64),
            EncoderBlock(64,  128),
            EncoderBlock(128, 256),
            EncoderBlock(256, 512),
            nn.Flatten(),
            nn.Linear(_BFLAT, latent_dim),
            nn.LayerNorm(latent_dim),
        )
        self.decoder_fc = nn.Sequential(
            nn.Linear(latent_dim, _BFLAT),
            nn.ReLU(inplace=True),
        )
        self.decoder_conv = nn.Sequential(
            DecoderBlock(512, 256, op=1),  # 19→38
            DecoderBlock(256, 128, op=1),  # 38→75
            DecoderBlock(128, 64),         # 75→150
            DecoderBlock(64,  32),         # 150→300
            nn.ConvTranspose2d(32, 1, 4, stride=2, padding=1),  # 300→600
            nn.Sigmoid(),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
                nn.init.kaiming_normal_(m.weight, nonlinearity='leaky_relu')
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def encode(self, x):
        """Encode image batch to latent vectors. Shape: (B,) → (B, latent_dim)"""
        return self.encoder(x)

    def decode(self, z):
        """Decode latent vectors to images. Shape: (B, latent_dim) → (B, 1, 600, 600)"""
        x = self.decoder_fc(z).view(-1, 512, 19, 19)
        return self.decoder_conv(x)

    def forward(self, x):
        return self.decode(self.encode(x))

    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


model = ConvAutoencoder(latent_dim=256).to(DEVICE)
print(f"Parameters: {model.count_params():,}")

# Sanity check
with torch.no_grad():
    dummy = torch.randn(2, 1, 600, 600, device=DEVICE)
    z = model.encode(dummy)
    r = model.decode(z)
print(f"Input: {dummy.shape} → z: {z.shape} → Recon: {r.shape}")

## 6. Loss Function (MSE + MS-SSIM)

In [ ]:
try:
    from pytorch_msssim import MS_SSIM
    _msssim = MS_SSIM(data_range=1.0, size_average=True, channel=1).to(DEVICE)
    HAS_SSIM = True
    print("MS-SSIM available")
except ImportError:
    HAS_SSIM = False
    print("MS-SSIM not available, using MSE only")

_mse = nn.MSELoss()
ALPHA = 0.15  # weight on MSE; (1-ALPHA) on MS-SSIM term

def combined_loss(recon, target):
    mse = _mse(recon, target)
    if HAS_SSIM:
        ssim_score = _msssim(recon, target)
        total = ALPHA * mse + (1 - ALPHA) * (1 - ssim_score)
        return total, mse.item(), ssim_score.item()
    return mse, mse.item(), 0.0

## 7. Training

In [ ]:
from tqdm.notebook import tqdm

EPOCHS = 50  # Increase to 100 for full training
LR     = 1e-3

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=5, factor=0.5, verbose=True)

history = {'train_loss': [], 'val_loss': [], 'val_mse': [], 'val_ssim': []}
best_val, best_epoch = float('inf'), 0

for epoch in range(1, EPOCHS + 1):
    # ── Train ──
    model.train()
    t_loss = 0.0
    for (imgs, _) in train_loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        loss, _, _ = combined_loss(model(imgs), imgs)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        t_loss += loss.item()
    t_loss /= len(train_loader)

    # ── Validate ──
    model.eval()
    v_loss = v_mse = v_ssim = 0.0
    with torch.no_grad():
        for (imgs, _) in val_loader:
            imgs = imgs.to(DEVICE, non_blocking=True)
            loss, mse, ssim = combined_loss(model(imgs), imgs)
            v_loss += loss.item(); v_mse += mse; v_ssim += ssim
    v_loss /= len(val_loader)
    v_mse  /= len(val_loader)
    v_ssim /= len(val_loader)
    scheduler.step(v_loss)

    history['train_loss'].append(t_loss)
    history['val_loss'].append(v_loss)
    history['val_mse'].append(v_mse)
    history['val_ssim'].append(v_ssim)

    if v_loss < best_val:
        best_val, best_epoch = v_loss, epoch
        torch.save({'model_state_dict': model.state_dict(),
                    'latent_dim': 256, 'epoch': epoch,
                    'val_loss': v_loss}, 'best_model.pth')

    if epoch % 5 == 0 or epoch == 1:
        print(f"Ep {epoch:>3}/{EPOCHS}  train={t_loss:.4f}  "
              f"val={v_loss:.4f}  MSE={v_mse:.4f}  SSIM={v_ssim:.4f}  "
              f"lr={optimizer.param_groups[0]['lr']:.1e}")

print(f"\nBest val loss {best_val:.5f} at epoch {best_epoch}")

## 8. Loss Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'],   label='Val')
axes[0].set_title('Combined Loss (MSE + MS-SSIM)')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history['val_mse'])
axes[1].set_title('Validation MSE')
axes[1].set_xlabel('Epoch')

axes[2].plot(history['val_ssim'], color='green')
axes[2].set_title('Validation MS-SSIM ↑')
axes[2].set_xlabel('Epoch')

plt.tight_layout()
plt.savefig('loss_curves.png', dpi=150)
plt.show()
print(f"Final val MSE: {history['val_mse'][-1]:.5f}")
print(f"Final val MS-SSIM: {history['val_ssim'][-1]:.4f}")

## 9. Reconstruction Quality

In [ ]:
# Load best model
ckpt = torch.load('best_model.pth', map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

samples, names = [], []
for i in range(min(6, len(dataset))):
    img, name = dataset[i]
    samples.append(img)
    names.append(name)
samples = torch.stack(samples).to(DEVICE)

with torch.no_grad():
    recons = model(samples)

fig, axes = plt.subplots(3, 6, figsize=(20, 10))
for i in range(6):
    orig  = samples[i, 0].cpu().numpy()
    recon = recons[i, 0].cpu().numpy()
    residual = np.abs(orig - recon)
    n_pl = names[i].split('_')[0].replace('planet', '')

    axes[0, i].imshow(orig,     cmap='inferno', origin='lower', vmin=0, vmax=1)
    axes[0, i].set_title(f'Input (N={n_pl})', fontsize=8)
    axes[1, i].imshow(recon,    cmap='inferno', origin='lower', vmin=0, vmax=1)
    axes[1, i].set_title(f'Reconstruction', fontsize=8)
    axes[2, i].imshow(residual, cmap='hot',    origin='lower')
    axes[2, i].set_title(f'|Residual|', fontsize=8)
    for row in range(3): axes[row, i].axis('off')

plt.suptitle('Input / Reconstruction / Residual', fontsize=14)
plt.tight_layout()
plt.savefig('reconstructions.png', dpi=150)
plt.show()

## 10. Latent Space Extraction

In [ ]:
model.eval()
all_latents, all_names, all_n_planets = [], [], []

full_loader = DataLoader(dataset, batch_size=8, shuffle=False, num_workers=2)

with torch.no_grad():
    for imgs, names_batch in full_loader:
        z = model.encode(imgs.to(DEVICE))
        all_latents.append(z.cpu().numpy())
        all_names.extend(names_batch)
        for n in names_batch:
            try:
                all_n_planets.append(int(n.split('_')[0].replace('planet', '')))
            except:
                all_n_planets.append(-1)

latents = np.vstack(all_latents)
n_planets_arr = np.array(all_n_planets)
print(f"Latent matrix: {latents.shape}")
print(f"Unique planet counts: {sorted(set(n_planets_arr))}")

## 11. UMAP + KMeans Clustering

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import umap

# Standardise latent vectors
scaler = StandardScaler()
latents_scaled = scaler.fit_transform(latents)

# UMAP 2D embedding
print("Running UMAP...")
reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
                    metric='euclidean', random_state=42)
embedding = reducer.fit_transform(latents_scaled)
print(f"Embedding shape: {embedding.shape}")

# KMeans — using planet-count heuristic for n_clusters
N_CLUSTERS = 5  # tune based on silhouette analysis below
km = KMeans(n_clusters=N_CLUSTERS, init='k-means++', n_init=20, random_state=42)
labels = km.fit_predict(latents_scaled)

sil = silhouette_score(latents_scaled, labels)
print(f"KMeans silhouette score: {sil:.4f}")

## 12. Silhouette Analysis — Finding Optimal k

In [ ]:
sil_scores = []
k_range = range(2, 12)
for k in k_range:
    km_tmp = KMeans(n_clusters=k, n_init=10, random_state=42)
    lbl_tmp = km_tmp.fit_predict(latents_scaled)
    sil_scores.append(silhouette_score(latents_scaled, lbl_tmp))

best_k = k_range[np.argmax(sil_scores)]
print(f"Best k={best_k} (silhouette={max(sil_scores):.4f})")

plt.figure(figsize=(7, 4))
plt.plot(list(k_range), sil_scores, 'o-', color='steelblue')
plt.axvline(best_k, color='red', linestyle='--', label=f'Best k={best_k}')
plt.xlabel('Number of clusters k')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Analysis for KMeans')
plt.legend()
plt.tight_layout()
plt.savefig('silhouette.png', dpi=150)
plt.show()

# Re-run with best k
km = KMeans(n_clusters=best_k, init='k-means++', n_init=20, random_state=42)
labels = km.fit_predict(latents_scaled)

## 13. UMAP Visualisation — Clusters vs. True Planet Count

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: coloured by KMeans cluster
scatter = axes[0].scatter(embedding[:, 0], embedding[:, 1],
                          c=labels, cmap='tab10', s=50,
                          edgecolors='k', linewidths=0.3, alpha=0.85)
plt.colorbar(scatter, ax=axes[0], label='Cluster')
axes[0].set_title(f'UMAP — KMeans (k={best_k})', fontsize=12)
axes[0].set_xlabel('UMAP 1'); axes[0].set_ylabel('UMAP 2')

# Right: coloured by true number of planets
sc2 = axes[1].scatter(embedding[:, 0], embedding[:, 1],
                      c=n_planets_arr, cmap='plasma', s=50,
                      edgecolors='k', linewidths=0.3, alpha=0.85)
plt.colorbar(sc2, ax=axes[1], label='N planets (ground truth)')
axes[1].set_title('UMAP — True Planet Count', fontsize=12)
axes[1].set_xlabel('UMAP 1'); axes[1].set_ylabel('UMAP 2')

plt.suptitle('ALMA Disk Latent Space — Unsupervised vs. Ground Truth', fontsize=13)
plt.tight_layout()
plt.savefig('umap_clusters.png', dpi=150)
plt.show()

## 14. Cluster Gallery

In [ ]:
N_SHOW = 4  # images per cluster
rng = np.random.default_rng(42)

cluster_ids = sorted(set(labels))
fig, axes = plt.subplots(len(cluster_ids), N_SHOW,
                         figsize=(4 * N_SHOW, 3.5 * len(cluster_ids)))

for row, cid in enumerate(cluster_ids):
    idx_in_cluster = np.where(labels == cid)[0]
    chosen = rng.choice(idx_in_cluster, size=min(N_SHOW, len(idx_in_cluster)), replace=False)
    for col in range(N_SHOW):
        ax = axes[row, col] if len(cluster_ids) > 1 else axes[col]
        if col < len(chosen):
            img, name = dataset[chosen[col]]
            n_pl = name.split('_')[0].replace('planet', '')
            ax.imshow(img[0].numpy(), cmap='afmhot', origin='lower')
            ax.set_title(f'N={n_pl}p', fontsize=8)
        ax.axis('off')
    axes[row, 0].set_ylabel(f'Cluster {cid}\n(n={len(idx_in_cluster)})',
                            fontsize=9, rotation=90, labelpad=8)

plt.suptitle('Cluster Gallery — ALMA Protoplanetary Disks', fontsize=13)
plt.tight_layout()
plt.savefig('cluster_gallery.png', dpi=150)
plt.show()

## 15. Latent Space Access Demo (Required by EXXA)

Demonstrating that a user can feed any image and access its encoded latent vector.

In [ ]:
# Example: encode a single image and retrieve its latent vector
model.eval()
sample_img, sample_name = dataset[0]
sample_tensor = sample_img.unsqueeze(0).to(DEVICE)  # (1, 1, 600, 600)

with torch.no_grad():
    z = model.encode(sample_tensor)       # (1, 256) — the latent code
    recon = model.decode(z)               # (1, 1, 600, 600)
    # or in one call:
    recon2 = model(sample_tensor)

print(f"Image: {sample_name}")
print(f"Latent vector shape: {z.shape}")
print(f"Latent vector (first 10 dims): {z[0, :10].cpu().numpy().round(4)}")

mse_val = F.mse_loss(recon, sample_tensor).item()
print(f"MSE (input vs reconstruction): {mse_val:.6f}")

## 16. Save All Results

In [ ]:
import json

results = {
    'n_samples': len(dataset),
    'n_clusters': int(best_k),
    'silhouette_score': float(silhouette_score(latents_scaled, labels)),
    'final_val_mse':  float(history['val_mse'][-1]),
    'final_val_ssim': float(history['val_ssim'][-1]),
    'best_epoch': int(best_epoch),
    'latent_dim': 256,
}
print(json.dumps(results, indent=2))
with open('results_summary.json', 'w') as f:
    json.dump(results, f, indent=2)
print("Saved results_summary.json")

# Files produced: best_model.pth, loss_curves.png, umap_clusters.png,
#                 cluster_gallery.png, reconstructions.png, silhouette.png

---
## Summary

| Metric | Value |
|--------|-------|
| Dataset | 150 synthetic ALMA continuum images (1250μm) |
| Architecture | ConvAutoencoder, 5-stage, latent dim=256 |
| Loss | α·MSE + (1-α)·(1-MS-SSIM), α=0.15 |
| Clustering | KMeans on UMAP-reduced latent space |
| Latent access | `model.encode(img)` → (1, 256) tensor |

**Code:** [GitHub — aryawadhwa](https://github.com/aryawadhwa/EXXA-Test)